In [1]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path().cwd() / "Data-LI"
LI_ACCOUNT_DATA_PATH = DATA_DIR / "LI-Small_accounts.csv"
LI_TRANSACTIONS_DATA_PATH = DATA_DIR / "LI-Small_Trans.csv"

accounts_df = pd.read_csv(LI_ACCOUNT_DATA_PATH)
trans_df = pd.read_csv(LI_TRANSACTIONS_DATA_PATH)

In [2]:
trans_df['Timestamp'] = pd.to_datetime(trans_df['Timestamp'])
trans_df = trans_df.rename(columns={'Account':'From Account', 'Account.1':'To Account'})
accounts_df = accounts_df.rename(columns={'Account Number': 'Account'})

accounts_df[
    'Universal_Account_ID'
    ] = accounts_df['Bank ID'].astype(str) + "_" + accounts_df['Account'].astype(str)
trans_df[
    'From_Universal_ID'
    ] = trans_df['From Bank'].astype(str) + "_" + trans_df['From Account'].astype(str)
trans_df[
    'To_Universal_ID'
         ] = trans_df['To Bank'].astype(str) + "_" + trans_df['To Account'].astype(str)

In [3]:
trans_df = trans_df.sort_values(by='Timestamp').reset_index(drop=True)

windows = ['1h', '3h', '6h', '12h', '24h']

temp_df = trans_df.set_index('Timestamp')

grouped_from = temp_df.groupby('From_Universal_ID')['Amount Paid']
grouped_to = temp_df.groupby('To_Universal_ID')['Amount Paid']
for window in windows:    
    roll_from = grouped_from.rolling(window).agg(['sum', 'count']).reset_index()
    
    roll_from = roll_from.rename(columns={
        'sum': f'from_vol_{window}',
        'count': f'from_count_{window}'
    })
    
    roll_from = roll_from.drop_duplicates(subset=['From_Universal_ID', 'Timestamp'], keep='last')
    trans_df = trans_df.merge(roll_from, on=['From_Universal_ID', 'Timestamp'], how='left')
    
    roll_to = grouped_to.rolling(window).agg(['sum', 'count']).reset_index()
    
    roll_to = roll_to.rename(columns={
        'sum': f'to_vol_{window}',
        'count': f'to_count_{window}'
    })
    
    roll_to = roll_to.drop_duplicates(subset=['To_Universal_ID', 'Timestamp'], keep='last')
    trans_df = trans_df.merge(roll_to, on=['To_Universal_ID', 'Timestamp'], how='left')

    roll_received_by_sender = roll_to.rename(columns={
        'To_Universal_ID': 'From_Universal_ID',
        f'to_vol_{window}': f'sender_received_vol_{window}',
        f'to_count_{window}': f'sender_received_count_{window}'
    })
    
    trans_df = trans_df.merge(
        roll_received_by_sender, 
        on=['From_Universal_ID', 'Timestamp'], 
        how='left'
    )
    
    trans_df[f'sender_received_vol_{window}'] = trans_df[f'sender_received_vol_{window}'].fillna(0)
    
    trans_df[f'pass_through_ratio_{window}'] = (
        trans_df[f'from_vol_{window}'] / (trans_df[f'sender_received_vol_{window}'] + 1)
    )

In [4]:
cols_to_clean = [col for col in trans_df.columns if 'Entity Name' in col or 'Universal_Account_ID' in col]
trans_df = trans_df.drop(columns=cols_to_clean, errors='ignore')

trans_df = trans_df.merge(
    accounts_df[['Universal_Account_ID', 'Entity Name']], 
    left_on='From_Universal_ID', 
    right_on='Universal_Account_ID', 
    how='left'
)

trans_df = trans_df.drop(columns=['Universal_Account_ID'])

print(trans_df.info())

<class 'pandas.DataFrame'>
RangeIndex: 6924049 entries, 0 to 6924048
Data columns (total 49 columns):
 #   Column                     Dtype         
---  ------                     -----         
 0   Timestamp                  datetime64[us]
 1   From Bank                  int64         
 2   From Account               str           
 3   To Bank                    int64         
 4   To Account                 str           
 5   Amount Received            float64       
 6   Receiving Currency         str           
 7   Amount Paid                float64       
 8   Payment Currency           str           
 9   Payment Format             str           
 10  Is Laundering              int64         
 11  From_Universal_ID          str           
 12  To_Universal_ID            str           
 13  from_vol_1h                float64       
 14  from_count_1h              float64       
 15  to_vol_1h                  float64       
 16  to_count_1h                float64       
 17  

In [5]:
trans_df['Entity_Type'] = trans_df['Entity Name'].str.replace(r'\s*#\d+', '', regex=True)

trans_df = trans_df.drop(columns=['Entity Name'])

print(trans_df['Entity_Type'].value_counts(normalize=True))

Entity_Type
Partnership            0.358747
Sole Proprietorship    0.343523
Corporation            0.295816
Individual             0.001913
Name: proportion, dtype: float64


In [6]:
import category_encoders as ce

count_enc = ce.CountEncoder(cols=['Entity_Type'], normalize=True)

trans_df['Entity_Frequency'] = count_enc.fit_transform(trans_df['Entity_Type'])

trans_df = trans_df.drop(columns=['Entity_Type'])

print("Distribuição das Frequências (Contexto Injetado):")
print(trans_df['Entity_Frequency'].value_counts().head())

Distribuição das Frequências (Contexto Injetado):
Entity_Frequency
0.358747    2483985
0.343523    2378573
0.295816    2048242
0.001913      13249
Name: count, dtype: int64


In [7]:
df_recebimentos = trans_df[['Timestamp', 'To_Universal_ID']].rename(
    columns={'To_Universal_ID': 'Account', 'Timestamp': 'Last_Received_Time'}
).sort_values('Last_Received_Time')

df_envios = trans_df[['Timestamp', 'From_Universal_ID']].rename(
    columns={'From_Universal_ID': 'Account', 'Timestamp': 'Send_Time'}
)

df_envios['orig_index'] = df_envios.index
df_envios = df_envios.sort_values('Send_Time')

df_rest_time = pd.merge_asof(
    df_envios,
    df_recebimentos,
    by='Account',
    left_on='Send_Time',
    right_on='Last_Received_Time',
    direction='backward'
)

df_rest_time['Rest_Time_Hours'] = (
    df_rest_time['Send_Time'] - df_rest_time['Last_Received_Time']
).dt.total_seconds() / 3600

trans_df['Rest_Time_Hours'] = df_rest_time.set_index('orig_index')['Rest_Time_Hours']

trans_df['Rest_Time_Hours'] = trans_df['Rest_Time_Hours'].fillna(-1)

In [8]:
for window in windows:
    trans_df[f'from_avg_ticket_{window}'] = (
        trans_df[f'from_vol_{window}'] / trans_df[f'from_count_{window}'].replace(0, 1)
    )

In [ ]:
import category_encoders as ce

categorical_cols = ['Receiving Currency', 'Payment Currency', 'Payment Format']

cat_encoder = ce.CountEncoder(cols=categorical_cols, normalize=True)

trans_df[categorical_cols] = cat_encoder.fit_transform(trans_df[categorical_cols])

In [14]:
metadata_cols = [
    'Timestamp', 'From Bank', 'From Account', 'To Bank', 'To Account', 
    'From_Universal_ID', 'To_Universal_ID', 'Is Laundering', 'Entity Name'
]

vol_cols = [col for col in trans_df.columns if '_vol_' in col]

features = [col for col in trans_df.columns if col not in metadata_cols and col not in vol_cols]

X = trans_df[features]

In [17]:
janelas_remover = ['_3h', '_6h', '_12h']

features_finais = [
    col for col in trans_df.columns 
    if col not in metadata_cols 
    and col not in vol_cols 
    and not any(j in col for j in janelas_remover)
]

X = trans_df[features_finais]

In [18]:
X.info()

<class 'pandas.DataFrame'>
RangeIndex: 6924049 entries, 0 to 6924048
Data columns (total 17 columns):
 #   Column                     Dtype  
---  ------                     -----  
 0   Amount Received            float64
 1   Receiving Currency         float64
 2   Amount Paid                float64
 3   Payment Currency           float64
 4   Payment Format             float64
 5   from_count_1h              float64
 6   to_count_1h                float64
 7   sender_received_count_1h   float64
 8   pass_through_ratio_1h      float64
 9   from_count_24h             float64
 10  to_count_24h               float64
 11  sender_received_count_24h  float64
 12  pass_through_ratio_24h     float64
 13  Entity_Frequency           float64
 14  Rest_Time_Hours            float64
 15  from_avg_ticket_1h         float64
 16  from_avg_ticket_24h        float64
dtypes: float64(17)
memory usage: 898.0 MB


In [20]:
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, confusion_matrix

y_true = trans_df['Is Laundering']

iso_forest = IsolationForest(
    n_estimators=300,        
    max_samples=20000,       
    contamination=0.05,      
    random_state=42, 
    n_jobs=-1
)

predictions = iso_forest.fit_predict(X)

y_pred = [1 if x == -1 else 0 for x in predictions]

print("\nMatriz de Confusão:")
print(confusion_matrix(y_true, y_pred))

print("\nRelatório de Classificação:")
print(classification_report(y_true, y_pred, target_names=['Normal (0)', 'Lavagem (1)']))

Treinando o modelo com hiperparâmetros ajustados...



Matriz de Confusão:
[[6574703  345781]
 [   3143     422]]

Relatório de Classificação:
              precision    recall  f1-score   support

  Normal (0)       1.00      0.95      0.97   6920484
 Lavagem (1)       0.00      0.12      0.00      3565

    accuracy                           0.95   6924049
   macro avg       0.50      0.53      0.49   6924049
weighted avg       1.00      0.95      0.97   6924049

